# Privacy Auditing with One Training Run — Full JupyterLab Notebook

**Paper:** Steinke, Nasr & Jagielski (2023) — *Privacy Auditing with One (1) Training Run*

**This notebook is fully self-contained** — no external `.py` modules required.  
It runs on JupyterLab with unlimited resources (multi-GPU optional).

### What this notebook does
1. Downloads CIFAR-10 automatically  
2. Trains **N shadow models** with Differentially Private SGD (Opacus)  
3. Computes membership-inference scores for canary examples  
4. Runs the statistical audit to produce a **provable ε lower bound**  
5. Produces publication-quality plots

### Config to tune (§ 2 — Configuration)
All hyperparameters live in one cell. Start small (`NUM_SHADOW=2`, `NUM_EPOCHS=5`) to verify the pipeline, then scale up.

> **Network / SSL note (IITGN & similar institutional servers)**  
> Your JupyterLab server sits behind a TLS-intercepting proxy that re-signs
> outbound HTTPS with an `*.iitgn.ac.in` certificate.  `pip` rejects this
> because it expects `pypi.org`.  
> **The install cell below detects this automatically and retries with
> `--trusted-host` flags.**  If it still fails, run the one-liner shown in
> the error message from a terminal *before* starting JupyterLab, or use the
> `conda` alternative shown there.  
> All other cells are network-free once packages are installed.

In [1]:
import resource

# Try to force the notebook to raise its own ceiling
try:
    soft, hard = resource.getrlimit(resource.RLIMIT_NOFILE)
    resource.setrlimit(resource.RLIMIT_NOFILE, (hard, hard))
    new_soft, _ = resource.getrlimit(resource.RLIMIT_NOFILE)
    print(f"Successfully raised limit from {soft} to {new_soft}")
except Exception as e:
    print(f"Could not raise limit: {e}")

Successfully raised limit from 4096 to 524288


In [2]:
CFG = {
    "data_dir": "/data/govindu.srimaan/cifar10"
}

In [3]:
# ── 1. Install / verify dependencies ─────────────────────────────────────────
# This cell handles two common network environments:
#   (a) Normal internet access  →  plain pip install
#   (b) Institutional proxy (e.g. IITGN) that re-signs TLS with its own cert
#       →  pip install --trusted-host + index fallback
#
# The cell tries a normal install first; if it fails due to an SSL error it
# retries with --trusted-host flags and, optionally, a mirror URL.

import subprocess, sys, os

# ── User config ───────────────────────────────────────────────────────────────
# If your institute provides a local PyPI mirror, put its URL here:
INSTITUTE_MIRROR = None   # e.g. "https://pypi.iitgn.ac.in/simple"
# Leave None to fall back to the global PyPI with trust flags.

PACKAGES = [
    "opacus>=1.5",
    "torch>=2.3",
    "torchvision>=0.18",
    "scipy>=1.13",
    "tqdm",
    "matplotlib",
]

# ─────────────────────────────────────────────────────────────────────────────
def _run(cmd):
    return subprocess.run(cmd, capture_output=True, text=True)

def pip_install_normal(pkgs):
    return _run([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

def pip_install_trusted(pkgs, mirror=None):
    """Install with SSL hostname trust bypass for institutional proxies."""
    trusted_hosts = [
        "--trusted-host", "pypi.org",
        "--trusted-host", "pypi.python.org",
        "--trusted-host", "files.pythonhosted.org",
    ]
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + trusted_hosts + pkgs
    if mirror:
        cmd += ["--index-url", mirror]
    return _run(cmd)

# ── Step 1: try normal install ────────────────────────────────────────────────
print("Trying standard pip install …")
result = pip_install_normal(PACKAGES)

if result.returncode == 0:
    print("✓ All packages installed / already present.")
else:
    stderr = result.stderr + result.stdout
    if "SSLError" in stderr or "CertificateError" in stderr or "hostname" in stderr.lower():
        print("SSL/certificate issue detected (institutional proxy).")
        print("Retrying with --trusted-host flags …")
        result2 = pip_install_trusted(PACKAGES, mirror=INSTITUTE_MIRROR)
        if result2.returncode == 0:
            print("✓ Packages installed via trusted-host bypass.")
        else:
            print("Second attempt also failed. Error output:")
            print(result2.stderr[-2000:])
            print()
            print("── Manual fix options ───────────────────────────────────────")
            print("Option A: run in a terminal before launching JupyterLab:")
            print("  pip install --trusted-host pypi.org --trusted-host files.pythonhosted.org \\")
            print("    opacus>=1.5 torch>=2.3 torchvision>=0.18 scipy matplotlib tqdm")
            print()
            print("Option B: use conda (avoids pip SSL issues entirely):")
            print("  conda install -c pytorch -c conda-forge pytorch torchvision scipy matplotlib tqdm")
            print("  pip install --trusted-host pypi.org opacus>=1.5")
            print()
            print("Option C: set INSTITUTE_MIRROR in this cell to your institute PyPI mirror URL.")
            raise RuntimeError("Dependency installation failed — see instructions above.")
    else:
        print("Installation failed for a non-SSL reason. Error:")
        print(result.stderr[-2000:])
        raise RuntimeError("pip install failed.")

# ── Verify imports ────────────────────────────────────────────────────────────
import torch, torchvision, opacus, scipy, tqdm, matplotlib
print(f"  torch       {torch.__version__}")
print(f"  torchvision {torchvision.__version__}")
print(f"  opacus      {opacus.__version__}")
print(f"  scipy       {scipy.__version__}")


Trying standard pip install …
✓ All packages installed / already present.
  torch       2.5.1+cu124
  torchvision 0.20.1+cu124
  opacus      1.5.4
  scipy       1.15.3


In [4]:
# ── 2. Imports and device ────────────────────────────────────────────────────
# Targeting Python 3.10 / JupyterHub (/opt/jupyterhub/bin/python)
import os, math, copy, random, json, pathlib, collections, warnings, time, sys
warnings.filterwarnings("ignore")

import numpy as np
import scipy.stats
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data
import torchvision
import torchvision.transforms as TF          # v1 API — always available
import torchvision.transforms.functional as TFF
import tqdm
import matplotlib.pyplot as plt
from opacus import PrivacyEngine
from opacus.utils.batch_memory_manager import BatchMemoryManager
from opacus.validators import ModuleValidator

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# ── torchvision.transforms.v2 compatibility shim ─────────────────────────────
# v2 was introduced in torchvision 0.15 and some servers have older versions.
# We define thin wrappers so the rest of the notebook uses a single API.
try:
    import torchvision.transforms.v2 as _T2
    _HAS_V2 = True
except ImportError:
    _HAS_V2 = False

class _Transforms:
    """Unified transform helpers that work on torch.Tensor inputs (uint8 or float)."""

    @staticmethod
    def to_float(x: torch.Tensor, dtype=torch.float32) -> torch.Tensor:
        """uint8 [0,255] → float [0,1]."""
        if x.dtype == torch.uint8:
            return x.to(dtype=dtype).div(255.0)
        return x.to(dtype=dtype)

    @staticmethod
    def normalize(x: torch.Tensor, mean, std) -> torch.Tensor:
        return TFF.normalize(x, mean=list(mean), std=list(std))

    @staticmethod
    def random_crop(x: torch.Tensor, size=32, padding=4) -> torch.Tensor:
        return TFF.pad(x, padding)
        # Then random crop
        _, h, w = x.shape
        padded = TFF.pad(x, padding)
        i = random.randint(0, 2 * padding)
        j = random.randint(0, 2 * padding)
        return padded[:, i:i+size, j:j+size]

    @staticmethod
    def random_hflip(x: torch.Tensor, p=0.5) -> torch.Tensor:
        if random.random() < p:
            return TFF.hflip(x)
        return x

    @staticmethod
    def to_image(x) -> torch.Tensor:
        """PIL Image → uint8 CHW tensor (or pass-through if already tensor)."""
        if isinstance(x, torch.Tensor):
            return x
        return TF.ToTensor()(x).mul(255).to(torch.uint8)

    @staticmethod
    def augment(x: torch.Tensor) -> torch.Tensor:
        x = _Transforms.random_crop(x, 32, 4)
        x = _Transforms.random_hflip(x)
        return x

TX = _Transforms()

# ── Device Setup for Parallel Shadow-Model Training ──────────────────────────

import os
import torch
import torchvision
import sys

# Select GPU manually per notebook/process
# Notebook 1 → GPU_ID = 0
# Notebook 2 → GPU_ID = 1

GPU_ID = 0

if torch.cuda.is_available():
    DEVICE = torch.device(f"cuda:{GPU_ID}")
    torch.cuda.set_device(GPU_ID)
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

DTYPE = torch.float32

NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0

# CIFAR-10 normalization constants
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2023, 0.1994, 0.2010)

print(f"Python        {sys.version.split()[0]}")
print(f"PyTorch       {torch.__version__}")
print(f"torchvision   {torchvision.__version__}")
print(f"Using Device  {DEVICE}")
print(f"CUDA GPUs     {NUM_GPUS}")

if torch.cuda.is_available():
    print(f"Current GPU   {torch.cuda.get_device_name(GPU_ID)}")

Python        3.10.12
PyTorch       2.5.1+cu124
torchvision   0.20.1+cu124
Using Device  cuda:0
CUDA GPUs     2
Current GPU   NVIDIA L40S


---
## § 2 — Configuration

Edit the values below. All other cells read from this one dict.

In [5]:
import torch
import numpy as np
import os
import pathlib

from datetime import datetime
experiment_name = f"blackbox_audit_v2_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

CFG = {
    # ── Experiment & Paths ──
    "experiment": experiment_name,
    "data_dir": pathlib.Path("./data"),
    "output_dir": pathlib.Path("./experiments"),
    
    # ── Audit (Algorithm 1) Parameters ──
    "num_canaries": 5000,        # Paper uses 3000–5000; we match at 5000
    "num_shadow": 16,            # Full audit
    "canary_type": "label_noise",
    "seed": 2024,

    # ── DP-SGD Training Parameters (CORRECTED FOR σ=0.4) ──
    "batch_size": 4096,          # Keep as is (paper uses this)
    "max_phys_batch": 1024,      # Memory manager chunk
    "num_workers": 8,
    
    # 🟢 KEY FIX: Reduce epochs from 150 → 60
    # Paper trains 60 epochs at σ=0.4. At 150 epochs, noise accumulates too much.
    "num_epochs": 60,
    
    # 🟢 KEY FIX: Learning rate tuned for 60 epochs + large batch
    # Paper uses LR=1.0–2.0 for batch_size=4096 at 60 epochs
    "learning_rate": 1.0,
    
    # 🟢 KEY FIX: Noise multiplier σ=0.4 (tight privacy, stronger signal)
    # Your σ=3.0 was too loose. Paper's main black-box result uses σ=0.4
    "noise_multiplier": 0.4,
    
    "max_grad_norm": 1.0,        # Standard; keep as is
    "delta": 1e-5,               # Standard; keep as is
    "device": "cuda" if torch.cuda.is_available() else "cpu",

    # ── Audit Evaluation Params ──
    "confidence_p": 0.05,        # 95% confidence
    "k_plus": 2500,              # Guess top 2500
    "k_minus": 2500,             # Guess bottom 2500
}

# Derived paths
CFG["data_dir"].mkdir(parents=True, exist_ok=True)
CFG["output_dir"].mkdir(parents=True, exist_ok=True)
CFG["ckpt_dir"] = CFG["output_dir"] / CFG["experiment"] / "shadow"
CFG["ckpt_dir"].mkdir(parents=True, exist_ok=True)

print("✅ Paper-Aligned Configuration Loaded!")
print(f"   σ={CFG['noise_multiplier']}, batch={CFG['batch_size']}, epochs={CFG['num_epochs']}, m={CFG['num_canaries']}")
print(f"   Expected theoretical ε ≤ ~100")
print(f"   Expected empirical ε ≥ 0.3–1.5 (black-box)")
print(f"   Checkpoints: {CFG['ckpt_dir']}")


# 🚀 FIX: Map global CKPT_DIR so the training loop never throws a NameError
CKPT_DIR = CFG["ckpt_dir"]

# 2. Lock Random Seeds
torch.manual_seed(CFG["seed"])
np.random.seed(CFG["seed"])

print("✅ Beast-Mode Configuration Loaded!")
print(f"✅ Targets: Batch={CFG['batch_size']}, PhysBatch={CFG['max_phys_batch']}, Epochs={CFG['num_epochs']}")
print(f"✅ Checkpoints will save to: {CKPT_DIR}")

✅ Paper-Aligned Configuration Loaded!
   σ=0.4, batch=4096, epochs=60, m=5000
   Expected theoretical ε ≤ ~100
   Expected empirical ε ≥ 0.3–1.5 (black-box)
   Checkpoints: experiments/blackbox_audit_v2_20260519_123930/shadow
✅ Beast-Mode Configuration Loaded!
✅ Targets: Batch=4096, PhysBatch=1024, Epochs=60
✅ Checkpoints will save to: experiments/blackbox_audit_v2_20260519_123930/shadow


In [6]:
import resource
soft, hard = resource.getrlimit(resource.RLIMIT_NOFILE)
print(f"Final Notebook Limit: {soft}")

Final Notebook Limit: 524288


---
## § 3 — Reproducibility & Seeds

In [7]:
# ── 4. Seed helpers ──────────────────────────────────────────────────────────
def setup_seeds(seed: int, deterministic: bool = True):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if deterministic:
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
        try:
            torch.use_deterministic_algorithms(True)
        except RuntimeError:
            pass  # some ops don't have deterministic kernels; skip silently
        os.environ["TF_CUDNN_DETERMINISTIC"] = "1"

def shadow_seed(global_seed: int, shadow_idx: int, num_shadow: int) -> int:
    return global_seed * (num_shadow + 1) + shadow_idx + 1

setup_seeds(CFG["seed"])
print(f"Global seed: {CFG['seed']}")

Global seed: 2024


In [8]:
import os
import tarfile
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import numpy as np
import random

# 1. SETUP PATHS - Based on your directory list, your data is likely in the 'data' folder
# We will check both the current directory and the 'data' subdirectory
POSSIBLE_DIRS = ["./", "./data"]
TAR_FILENAME = "cifar-10-python.tar.gz" 

found_path = None
for d in POSSIBLE_DIRS:
    test_path = os.path.join(d, TAR_FILENAME)
    if os.path.exists(test_path):
        found_path = d
        break

if found_path:
    DATA_DIR = found_path
    print(f"✅ Found {TAR_FILENAME} in: {DATA_DIR}")
else:
    # If not found, let's look at what is INSIDE the 'data' folder to help you
    print(f"❌ Could not find {TAR_FILENAME}")
    if os.path.exists("./data"):
        print("Contents of './data' folder:", os.listdir("./data"))
    DATA_DIR = "./data" # Defaulting to data folder

TAR_PATH = os.path.join(DATA_DIR, TAR_FILENAME)
EXTRACT_PATH = os.path.join(DATA_DIR, "cifar-10-batches-py")

# 2. EXTRACTION
if not os.path.exists(EXTRACT_PATH):
    if os.path.exists(TAR_PATH):
        print(f"Extracting {TAR_PATH}...")
        with tarfile.open(TAR_PATH, "r:gz") as tar:
            tar.extractall(path=DATA_DIR)
        print("Extraction complete.")
    else:
        print("Skipping extraction: Tar file not found or already extracted.")

# 3. TRANSFORMS
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# 4. LOAD DATASET
try:
    # IMPORTANT: root should be the folder that CONTAINS 'cifar-10-batches-py'
    trainset = torchvision.datasets.CIFAR10(root=DATA_DIR, train=True, download=False, transform=transform)
    testset  = torchvision.datasets.CIFAR10(root=DATA_DIR, train=False, download=False, transform=transform)
    print(f"🚀 SUCCESS: Dataset Loaded. Train size: {len(trainset)}")
except Exception as e:
    print(f"❌ Error loading dataset: {e}")
    print("Check if 'cifar-10-batches-py' exists inside your 'data' folder.")

✅ Found cifar-10-python.tar.gz in: ./data
🚀 SUCCESS: Dataset Loaded. Train size: 50000


In [9]:
import torchvision
import torchvision.transforms as transforms
import torch

print("Loading CIFAR-10 Dataset...")
# 1. Load the raw datasets
transform = transforms.ToTensor()
trainset = torchvision.datasets.CIFAR10(root=CFG["data_dir"], train=True, download=True, transform=transform)
testset  = torchvision.datasets.CIFAR10(root=CFG["data_dir"], train=False, download=True, transform=transform)

# 2. Extract raw tensors for the DataGenerator
print("Converting datasets to raw tensors...")
_train_xs = torch.stack([trainset[i][0] for i in range(len(trainset))])
_train_ys_clean = torch.tensor(trainset.targets)

_test_xs = torch.stack([testset[i][0] for i in range(len(testset))])
_test_ys = torch.tensor(testset.targets)

print(f"✅ Ready: _train_xs shape: {_train_xs.shape}")

Loading CIFAR-10 Dataset...
Files already downloaded and verified
Files already downloaded and verified
Converting datasets to raw tensors...
✅ Ready: _train_xs shape: torch.Size([50000, 3, 32, 32])


---
## § 4 — Dataset Generator (Canary Injection + IN/OUT Splits)

In [10]:
# ── 6. Canary + shadow-split generator ───────────────────────────────────────
class PrivacyAuditDataset(torch.utils.data.Dataset):
    def __init__(self, xs, ys, indices=None, transform=None):
        self.xs = xs
        self.ys = ys
        self.idx = (torch.arange(len(xs)) if indices is None
                    else torch.as_tensor(indices, dtype=torch.long))
        self.transform = transform

    def __len__(self):  return len(self.idx)

    def __getitem__(self, i):
        x = self.xs[self.idx[i]]
        y = self.ys[self.idx[i]]
        if self.transform:
            x = self.transform(x)
        return x, y

    def as_features_only(self, transform=None):
        return _FeaturesOnly(self.xs, self.idx, transform or self.transform)


class _FeaturesOnly(torch.utils.data.Dataset):
    def __init__(self, xs, idx, transform):
        self.xs = xs; self.idx = idx; self.transform = transform
    def __len__(self):  return len(self.idx)
    def __getitem__(self, i):
        x = self.xs[self.idx[i]]
        return self.transform(x) if self.transform else x


class DataGenerator:
    """Manages canary injection and per-shadow IN/OUT membership."""

    def __init__(self, train_xs, train_ys_clean, test_xs, test_ys,
                 num_shadow, num_canaries, canary_type, seed):
        self.train_xs       = train_xs
        self.train_ys_clean = train_ys_clean
        self.test_xs        = test_xs
        self.test_ys        = test_ys
        self.num_shadow     = num_shadow
        self.num_canaries   = num_canaries
        self.canary_type    = canary_type
        self.N              = len(train_xs)
        self.num_classes    = 10

        rng = np.random.default_rng(seed)

        # ── 1. IN/OUT splits (Algorithm 1: Independent Bernoulli Flips) ───────
        # Identify which samples are canaries
        self.canary_order = rng.permutation(self.N)
        canary_indices = self.canary_order[:num_canaries]
        
        # Start by asserting EVERY sample is IN the training set for all shadow models
        shadow_in = np.ones((self.N, num_shadow), dtype=bool)
        
        # For ONLY the canaries, flip an independent 50/50 coin for each shadow model
        # rng.integers(0, 2) generates independent 0s and 1s
        coin_flips = rng.integers(0, 2, size=(num_canaries, num_shadow)).astype(bool)
        shadow_in[canary_indices, :] = coin_flips
        
        self.shadow_in_mask = shadow_in     # (N, num_shadow) bool

        # ── 2. Noisy labels (used for label_noise canaries) ───────────────────
        label_changes = rng.integers(self.num_classes - 1, size=self.N)
        clean = train_ys_clean.numpy()
        self.noisy_ys = np.where(label_changes < clean, label_changes, label_changes + 1)

    # ------------------------------------------------------------------
    def get_canary_indices(self):
        return self.canary_order[: self.num_canaries]

    def _make_train_tensors(self):
        """Return (xs, ys) with canary modifications applied."""
        xs = self.train_xs.clone()
        ys = self.train_ys_clean.clone().numpy()
        ci = self.get_canary_indices()
        if self.canary_type == "label_noise":
            ys[ci] = self.noisy_ys[ci]
        elif self.canary_type == "blank_images":
            rng2 = np.random.default_rng(42)
            colors = rng2.integers(0, 256, size=(self.num_canaries, 3, 1, 1), dtype=np.uint8)
            xs[ci] = torch.from_numpy(
                np.broadcast_to(colors, (self.num_canaries, 3, 32, 32)).copy()
            )
            rand_labels = rng2.integers(0, self.num_classes, size=self.num_canaries)
            ys[ci] = rand_labels
        # "clean": no change
        return xs, torch.from_numpy(ys)

    def build_train_data(self, shadow_idx, transform=None):
        xs, ys = self._make_train_tensors()
        in_idx = np.where(self.shadow_in_mask[:, shadow_idx])[0]
        return PrivacyAuditDataset(xs, ys, indices=in_idx, transform=transform)

    def build_full_train_data(self, transform=None):
        xs, ys = self._make_train_tensors()
        return PrivacyAuditDataset(xs, ys, transform=transform)

    def build_test_data(self, transform=None):
        return PrivacyAuditDataset(self.test_xs, self.test_ys, transform=transform)

    def build_in_mask(self, shadow_idx):
        return torch.from_numpy(self.shadow_in_mask[:, shadow_idx])


# Instantiate
data_gen = DataGenerator(
    train_xs=_train_xs, train_ys_clean=_train_ys_clean,
    test_xs=_test_xs,   test_ys=_test_ys,
    num_shadow=CFG["num_shadow"],
    num_canaries=CFG["num_canaries"],
    canary_type=CFG["canary_type"],
    seed=CFG["seed"],
)
ci = data_gen.get_canary_indices()
print(f"Canary indices (first 10): {ci[:10]}")
print(f"Average canary IN count per shadow model: {data_gen.shadow_in_mask[ci].mean():.3f}")

Canary indices (first 10): [ 6160  9086 18826 31124 13332 13007 24878 49531 30951 15368]
Average canary IN count per shadow model: 0.500


---
## § 5 — WideResNet-16-4 (Group-Norm, Opacus-compatible)

In [11]:
# ── 7. WideResNet ────────────────────────────────────────────────────────────
class WideBlock(nn.Module):
    def __init__(self, fin, fout, stride, expand, norm_fn, swap=True):
        super().__init__()
        self.swap = swap
        self.n1 = norm_fn(fin)
        self.c1 = nn.Conv2d(fin, fout, 3, 1, 1)
        self.n2 = norm_fn(fout)
        self.c2 = nn.Conv2d(fout, fout, 3, stride, 1)
        if stride != 1 or expand:
            if swap:
                self.skip = nn.Sequential(nn.ReLU(), norm_fn(fin),
                                          nn.Conv2d(fin, fout, 1, stride))
            else:
                self.skip = nn.Conv2d(fin, fout, 1, stride)
        else:
            self.skip = nn.Identity()

    def forward(self, x):
        if self.swap:
            h = self.c1(self.n1(F.relu(x)))
            h = self.c2(self.n2(F.relu(h)))
        else:
            h = self.c1(F.relu(self.n1(x)))
            h = self.c2(F.relu(self.n2(h)))
        return h + self.skip(x)


class WideResNet(nn.Module):
    def __init__(self, depth=16, widen=4, num_classes=10, swap=True):
        super().__init__()
        assert (depth - 4) % 6 == 0
        nb = (depth - 4) // 6
        fs = (16, 16*widen, 32*widen, 64*widen)

        def norm(c): return nn.GroupNorm(min(16, c), c)

        self.conv0  = nn.Conv2d(3, fs[0], 3, 1, 1)
        self.group1 = self._layer(fs[0], fs[1], 1, nb, norm, swap)
        self.group2 = self._layer(fs[1], fs[2], 2, nb, norm, swap)
        self.group3 = self._layer(fs[2], fs[3], 2, nb, norm, swap)
        self.norm_out = norm(fs[3])
        self.pool   = nn.AdaptiveAvgPool2d(1)
        self.flat   = nn.Flatten()
        self.fc     = nn.Linear(fs[3], num_classes)
        self.swap   = swap
        self._fin   = fs[3]
        self._init()

    def _layer(self, fin, fout, stride, n, norm_fn, swap):
        blocks = []
        for i in range(n):
            blocks.append(WideBlock(fin if i==0 else fout, fout,
                                    stride if i==0 else 1,
                                    fin != fout, norm_fn, swap))
        return nn.Sequential(*blocks)

    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                fi = max(1, m.weight.size(1) * m.weight.size(2) * m.weight.size(3))
                nn.init.trunc_normal_(m.weight, std=1/fi**0.5)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.GroupNorm):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                fi = max(1, m.weight.size(1))
                nn.init.trunc_normal_(m.weight, std=1/fi**0.5)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        h = self.conv0(x)
        h = self.group1(h); h = self.group2(h); h = self.group3(h)
        h = self.norm_out(F.relu(h)) if self.swap else F.relu(self.norm_out(h))
        return self.fc(self.flat(self.pool(h)))

    def replace_head(self, num_classes):
        self.fc = nn.Linear(self._fin, num_classes).to(self.fc.weight.device)

def build_model(num_classes=10):
    return WideResNet(depth=16, widen=4, num_classes=num_classes, swap=True).to(DEVICE)

# Quick smoke-test
_m = build_model()
_x = torch.randn(2, 3, 32, 32, device=DEVICE)
assert _m(_x).shape == (2, 10)
print(f"WideResNet-16-4: {sum(p.numel() for p in _m.parameters()):,} parameters")
del _m, _x

WideResNet-16-4: 2,838,922 parameters


---
## § 6 — DP-SGD Training Loop

In [12]:
# ── 8. Transforms and dataset wrapper ────────────────────────────────────────
# All transforms operate on uint8 CHW tensors (as loaded into memory).
# TX helper (defined in cell 2) provides float-conversion, normalise, augment.

class NormDataset(torch.utils.data.Dataset):
    """
    Wraps PrivacyAuditDataset; converts uint8 → normalised float on-the-fly.
    Optionally applies random augmentation (for training).
    """
    def __init__(self, ds: PrivacyAuditDataset, augment: bool = False):
        self.ds      = ds
        self.augment = augment

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, i):
        x, y = self.ds[i]
        x = TX.to_float(x, DTYPE)
        if self.augment:
            x = TX.augment(x)
        x = TX.normalize(x, CIFAR10_MEAN, CIFAR10_STD)
        return x, y


class InferenceDataset(torch.utils.data.Dataset):
    """Features-only dataset with normalisation; used during inference."""
    def __init__(self, ds: PrivacyAuditDataset):
        self.ds = ds

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, i):
        x, _ = self.ds[i]
        x = TX.to_float(x, DTYPE)
        x = TX.normalize(x, CIFAR10_MEAN, CIFAR10_STD)
        return x


In [13]:
# ── 9. EMA helper ────────────────────────────────────────────────────────────
import copy

def create_ema(model):
    ema = copy.deepcopy(model)
    for p in ema.parameters():
        p.detach_()
    return ema

def update_ema(model, ema, t, decay=0.9999):
    t2 = t
    # Ramp-up strategy: faster updates early on, settling to 'decay'
    eff = min(decay, (1 + t2) / (10 + t2))
    mp = dict(model.named_parameters())
    ep = dict(ema.named_parameters())
    for name in mp:
        ep[name].sub_((1.0 - eff) * (ep[name] - mp[name]))
    
    # Buffers (like Batchnorm stats) are copied directly
    mb = dict(model.named_buffers())
    eb = dict(ema.named_buffers())
    for name in mb:
        eb[name].copy_(mb[name])

In [14]:
import json
import time
import copy
import warnings

def train_shadow_model(shadow_idx: int, cfg: dict,
                       data_gen: DataGenerator,
                       verbose: bool = True):
    """
    Train one shadow model with DP-SGD, saving continuous progress.
    """
    ckpt_dir  = cfg["ckpt_dir"] / str(shadow_idx)
    ckpt_ema  = ckpt_dir / "ema_model.pt"
    ckpt_w0   = ckpt_dir / "w0_state.pt"
    stats_file = ckpt_dir / "training_log.json"

    if ckpt_ema.exists() and ckpt_w0.exists():
        print(f"  [shadow {shadow_idx}] Checkpoint found — loading.")
        ema   = torch.load(ckpt_ema, map_location=DEVICE, weights_only=False)
        w0_sd = torch.load(ckpt_w0,  map_location="cpu",  weights_only=True)
        return ema, w0_sd

    ckpt_dir.mkdir(parents=True, exist_ok=True)
    setup_seeds(shadow_seed(cfg["seed"], shadow_idx, cfg["num_shadow"]))

    # ── Dataset ──────────────────────────────────────────────────────────────
    raw_ds  = data_gen.build_train_data(shadow_idx)
    train_ds = raw_ds
    loader   = torch.utils.data.DataLoader(
        train_ds,
        batch_size=cfg["batch_size"],
        shuffle=True,
        num_workers=cfg["num_workers"],
        drop_last=False,
        persistent_workers=(cfg["num_workers"] > 0),
        pin_memory=True,
    )

    # ── Model (single GPU for Opacus compatibility) ──────────────────────────
    base_model = WideResNet(depth=16, widen=4, num_classes=10).to(DEVICE)
    model = ModuleValidator.fix(base_model)
    errs  = ModuleValidator.validate(model, strict=False)
    if errs:
        raise RuntimeError(f"Opacus validation errors: {errs}")

    # ── Save w_0 ─────────────────────────────────────────────────────────────
    w0_state = copy.deepcopy(model.state_dict())
    w0_state_cpu = {k: v.cpu() for k, v in w0_state.items()}
    torch.save(w0_state_cpu, ckpt_w0)

    # ── Optimiser + Opacus ───────────────────────────────────────────────────
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=cfg["learning_rate"],
        momentum=0.0,
        weight_decay=0.0,
    )
    loss_fn = nn.CrossEntropyLoss(reduction="mean")

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        pe = PrivacyEngine()
        model, optimizer, loader = pe.make_private(
            module=model,
            optimizer=optimizer,
            data_loader=loader,
            noise_multiplier=cfg["noise_multiplier"],
            max_grad_norm=cfg["max_grad_norm"],
        )

    ema        = create_ema(model)
    num_updates = 0
    log_every  = max(1, cfg["num_epochs"] // 10)
    
    training_history = []

    # ── Training loop ─────────────────────────────────────────────────────────
    for epoch in range(cfg["num_epochs"]):
        model.train()
        with BatchMemoryManager(
            data_loader=loader,
            max_physical_batch_size=cfg["max_phys_batch"],
            optimizer=optimizer,
        ) as opt_loader:
            epoch_loss = epoch_n = 0

            for bx, by in opt_loader:
                bx = bx.to(DEVICE)
                by = by.to(DEVICE)
                optimizer.zero_grad(set_to_none=True)

                loss = loss_fn(model(bx), by)
                loss.backward()

                will_update = not optimizer._check_skip_next_step(pop_next=False)
                optimizer.step()
                if will_update:
                    num_updates += 1
                    update_ema(model, ema, num_updates)

                epoch_loss += loss.item() * bx.size(0)
                epoch_n    += bx.size(0)

        # ── INTERMEDIATE SAVING LOGIC ──
        eps = pe.get_epsilon(cfg["delta"])
        avg_loss = epoch_loss / epoch_n
        
        # 1. Save stats to JSON
        training_history.append({"epoch": epoch + 1, "loss": avg_loss, "epsilon": eps})
        with open(stats_file, "w") as f:
            json.dump(training_history, f, indent=4)

        # 2. Overwrite latest backup
        torch.save(ema, ckpt_dir / "ema_latest.pt")
        
        # 3. Save Milestone checkpoints (e.g., at epoch 50, 100)
        if (epoch + 1) % 50 == 0:
            torch.save(ema, ckpt_dir / f"ema_epoch_{epoch+1}.pt")

        if verbose or (epoch + 1) % log_every == 0:
            print(f"    epoch {epoch+1:3d}/{cfg['num_epochs']}  loss={avg_loss:.4f}  ε={eps:.4f}")

    # Final Save
    ema.eval()
    torch.save(ema, ckpt_ema)
    print(f"  [shadow {shadow_idx}] finished & saved → {ckpt_dir}")
    return ema, w0_state_cpu


---
## § 7 — Train All Shadow Models

In [15]:
CKPT_DIR = CFG["ckpt_dir"]

In [16]:
# ── Train shadow models on GPU 0 ─────────────────────────────────────────────

trained_models = {}
t0 = time.time()

START_IDX = 8
END_IDX   = 16  # Remember to change this to 16-32 on GPU 1!

for s in range(START_IDX, END_IDX):
    print(f"\n─── Shadow model {s+1}/{CFG['num_shadow']} ───")
    trained_models[s] = train_shadow_model(s, CFG, data_gen, verbose=True)

print(f"\nGPU finished models {START_IDX}-{END_IDX-1} in {time.time()-t0:.0f}s")


─── Shadow model 9/16 ───


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.13 GiB. GPU 0 has a total capacity of 44.40 GiB of which 1.03 GiB is free. Process 1087728 has 30.01 GiB memory in use. Including non-PyTorch memory, this process has 13.33 GiB memory in use. Of the allocated memory 11.20 GiB is allocated by PyTorch, and 1.58 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

---
## § 8 — Score Computation

In [ ]:
# ── 12. Inference helper ─────────────────────────────────────────────────────
import gc

@torch.no_grad()
def predict_logits(model: nn.Module, dataset, batch_size=512) -> torch.Tensor:
    """
    Run model on `dataset` (features only) and return logits shape (N, C).
    """
    model.eval()
    
    # 🟢 FIX: persistent_workers=False prevents "Too Many Open Files" error
    loader = torch.utils.data.DataLoader(
        dataset, 
        batch_size=batch_size, 
        shuffle=False,
        num_workers=CFG["num_workers"],
        persistent_workers=False, 
        pin_memory=True
    )
    
    out = []
    for bx in loader:
        # bx is moved to the specific GPU device defined in your Setup cell
        out.append(model(bx.to(DEVICE)).cpu())
    
    logits = torch.cat(out, dim=0)

    # 🟢 FIX: Explicitly shut down workers and clear memory
    del loader
    gc.collect()
    
    return logits

In [ ]:
# ── 13. Compute membership scores (CORRECTED: use loss(w_0, x) - loss(w_T, x)) ──
import os
import gc

print("Building full-train dataset (with canary labels applied) …")
full_ds = data_gen.build_full_train_data()
print("\n── LABEL DIAGNOSTIC ──")

for idx in canary_idx[:10]:
    idx = idx.item()

    x, y = full_ds[idx]

    print(f"Canary {idx}: label in full_ds = {y}")
feat_ds = InferenceDataset(full_ds)
all_ys = torch.stack([full_ds[i][1] for i in range(len(full_ds))])

canary_idx = torch.from_numpy(data_gen.get_canary_indices().astype("int64"))
N_CANARY   = len(canary_idx)
print(f"Scoring {N_CANARY} canaries across {CFG['num_shadow']} shadow models …\n")

shadow_scores = torch.zeros(CFG["num_shadow"], N_CANARY)
membership    = torch.zeros(CFG["num_shadow"], N_CANARY, dtype=torch.bool)

# 🟢 FIX: Load BOTH w_0 and w_T for each model
for s in range(CFG["num_shadow"]):
    ckpt_base = CFG["ckpt_dir"] / str(s)
    ckpt_ema  = ckpt_base / "ema_latest.pt"  # or "ema_model.pt" if that's your file
    ckpt_w0   = ckpt_base / "w0_state.pt"
    
    # Verify both files exist
    if not ckpt_ema.exists():
        print(f"  [Warning] Shadow {s:2d} missing EMA at {ckpt_ema}! Skipping...")
        continue
    if not ckpt_w0.exists():
        print(f"  [Warning] Shadow {s:2d} missing w0 at {ckpt_w0}! Skipping...")
        continue
        
    print(f"  Shadow {s:2d}: loading w_0 and w_T … ", end="", flush=True)
    
    # ── Load w_0 (initial weights) ──
    w0_state = torch.load(ckpt_w0, map_location="cpu", weights_only=True)
    w0_model = build_model()  # Build fresh model
    w0_model.load_state_dict(w0_state, strict=True)
    w0_model.eval()
    
    # Compute loss(w_0, x) for all 50k examples
    initial_logits = predict_logits(w0_model, feat_ds, batch_size=512)
    initial_loss = F.cross_entropy(initial_logits, all_ys, reduction="none").detach()
    
    del w0_model, initial_logits
    gc.collect()
    torch.cuda.empty_cache()
    
    # ── Load w_T (trained EMA weights) ──
    final_model = torch.load(ckpt_ema, map_location=CFG["device"], weights_only=False)
    final_model.eval()
    
    # Compute loss(w_T, x) for all 50k examples
    final_logits = predict_logits(final_model, feat_ds, batch_size=512)
    final_loss = F.cross_entropy(final_logits, all_ys, reduction="none").detach()
    
    del final_model, final_logits
    gc.collect()
    torch.cuda.empty_cache()
    
    # ── 🟢 CORRECTED SCORE: loss(w_0, x) - loss(w_T, x) ──
    score_full = initial_loss - final_loss
    
    shadow_scores[s] = score_full[canary_idx].cpu()
    membership[s]    = data_gen.build_in_mask(s)[canary_idx]
        
    nin  = membership[s].sum().item()
    nout = N_CANARY - nin
    s_in  = shadow_scores[s][membership[s]].mean().item()
    s_out = shadow_scores[s][~membership[s]].mean().item()
    print(f"IN={nin:4d}  OUT={nout:4d}  "
          f"score_in={s_in:+.4f}  score_out={s_out:+.4f}  Δ={s_in-s_out:+.4f}")

print("\nScoring complete.")

# ── SAVE SCORES AND MEMBERSHIP ──
scores_path = CFG["ckpt_dir"] / "shadow_scores.npy"
member_path = CFG["ckpt_dir"] / "membership.npy"
np.save(scores_path, shadow_scores.numpy())
np.save(member_path, membership.numpy())
print(f"Saved scores     → {scores_path}")
print(f"Saved membership → {member_path}")

In [ ]:
import os
from pathlib import Path

base = CFG["ckpt_dir"]

print(base)

for s in range(CFG["num_shadow"]):
    p = base / str(s)

    print(f"\nShadow {s}")

    if p.exists():
        print("  Exists")

        for f in p.iterdir():
            print("   ", f.name)
    else:
        print("  Missing directory")

In [ ]:
# ── DIAGNOSTIC: Inspect score distributions ──────────────────────────────────
import matplotlib.pyplot as plt

print("=" * 70)
print("SCORE DIAGNOSTIC")
print("=" * 70)

# Look at shadow 0 in detail
s = 0
scores_s = shadow_scores[s]
member_s = membership[s]

print(f"\nShadow {s} score statistics:")
print(f"  All scores:      min={scores_s.min():.4f}, max={scores_s.max():.4f}, mean={scores_s.mean():.4f}")
print(f"  IN scores:       min={scores_s[member_s].min():.4f}, max={scores_s[member_s].max():.4f}, mean={scores_s[member_s].mean():.4f}")
print(f"  OUT scores:      min={scores_s[~member_s].min():.4f}, max={scores_s[~member_s].max():.4f}, mean={scores_s[~member_s].mean():.4f}")
print(f"  Δ mean (IN-OUT): {scores_s[member_s].mean() - scores_s[~member_s].mean():.4f}")

# Check if all scores are zero (would indicate a loading or computation bug)
if scores_s.abs().sum() < 1e-6:
    print("\n⚠️  WARNING: All scores are zero! Check if w_0 and w_T are being loaded correctly.")
else:
    print(f"\n✓ Scores have nonzero variance: {scores_s.var():.4f}")

# Histogram
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(scores_s[member_s].numpy(), bins=50, alpha=0.6, label="IN", density=True)
ax.hist(scores_s[~member_s].numpy(), bins=50, alpha=0.6, label="OUT", density=True)
ax.set_xlabel("Score = loss(w_0, x) - loss(w_T, x)")
ax.set_ylabel("Density")
ax.set_title(f"Score Distribution (Shadow {s})")
ax.legend()
ax.axvline(0, color='red', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig("score_diagnostic.png", dpi=100)
plt.show()

print(f"\nPlot saved to score_diagnostic.png")
print("=" * 70)

In [ ]:
# ── SANITY CHECK: Verify loss computation ────────────────────────────────────
print("\n" + "=" * 70)
print("LOSS SANITY CHECK")
print("=" * 70)

s = 0
ckpt_base = CFG["ckpt_dir"] / str(s)
ckpt_w0   = ckpt_base / "w0_state.pt"
ckpt_ema  = ckpt_base / "ema_latest.pt"

# Load w_0
w0_state = torch.load(ckpt_w0, map_location="cpu", weights_only=True)
w0_model = build_model()
w0_model.load_state_dict(w0_state, strict=True)
w0_model.eval()

# Load w_T
ema_model = torch.load(ckpt_ema, map_location=CFG["device"], weights_only=False)
ema_model.eval()

# Compute losses on a SMALL SAMPLE (first 100 canaries)
test_idx = canary_idx[:100]
test_xs = torch.stack([feat_ds[int(i)] for i in test_idx])  # (100, 3, 32, 32)
test_ys = all_ys[test_idx]  # (100,)

with torch.no_grad():
    w0_logits = w0_model(test_xs.to(DEVICE))
    w0_loss = F.cross_entropy(w0_logits, test_ys.to(DEVICE), reduction="none").cpu()
    
    ema_logits = ema_model(test_xs.to(DEVICE))
    ema_loss = F.cross_entropy(ema_logits, test_ys.to(DEVICE), reduction="none").cpu()

score_sample = w0_loss - ema_loss

print(f"\nOn {len(test_idx)} canary examples:")
print(f"  w_0 loss:    min={w0_loss.min():.4f}, max={w0_loss.max():.4f}, mean={w0_loss.mean():.4f}")
print(f"  w_T loss:    min={ema_loss.min():.4f}, max={ema_loss.max():.4f}, mean={ema_loss.mean():.4f}")
print(f"  Score:       min={score_sample.min():.4f}, max={score_sample.max():.4f}, mean={score_sample.mean():.4f}")

# Check: if w_T is randomly initialized, it should have similar loss to w_0
# If w_T was trained, loss should be lower → score should be positive
print(f"\n  Expected: w_T loss < w_0 loss (because model was trained)")
print(f"  Actual:   w_T mean ({ema_loss.mean():.4f}) vs w_0 mean ({w0_loss.mean():.4f})")
print(f"  Training helped? {ema_loss.mean() < w0_loss.mean()}")

print("=" * 70)

---
## § 9 — Score Aggregation & Audit Statistics

In [ ]:
# ── 14. Statistical Math Helpers for Algorithm 1 ─────────────────────────────
import math
import scipy.stats

def p_value_DP_audit(m, r, v, eps, delta):
    """
    P-value of ≥v correct guesses under the null-hypothesis (eps,delta)-DP model.
    Corollary 5.4 of Steinke et al. 2023.
    """
    assert 0 <= v <= r <= m and eps >= 0 and 0 <= delta <= 1
    q = 1.0 / (1.0 + math.exp(-eps))
    beta = scipy.stats.binom.sf(v - 1, r, q)
    alpha, running = 0.0, 0.0
    for i in range(1, v + 1):
        running += scipy.stats.binom.pmf(v - i, r, q)
        if running > i * alpha:
            alpha = running / i
    return min(beta + alpha * delta * 2 * m, 1.0)


def get_eps_lower_bound(m, r, v, delta, p=0.05):
    """Binary-search for the largest ε such that p_value < p."""
    assert 0 <= v <= r <= m and 0 < p < 1
    eps_lo, eps_hi = 0.0, 1.0
    
    # Find upper bound for binary search
    while p_value_DP_audit(m, r, v, eps_hi, delta) < p:
        eps_hi += 1.0
        
    # Binary search for exact epsilon
    for _ in range(50):
        eps_mid = (eps_lo + eps_hi) / 2.0
        if p_value_DP_audit(m, r, v, eps_mid, delta) < p:
            eps_lo = eps_mid
        else:
            eps_hi = eps_mid
            
    return eps_lo

In [ ]:
# ── 14. Inject Auditing Hyperparameters into CFG ─────────────────────────────
CFG["k_plus"] = 1000       # Reasonable default for 5000 canaries
CFG["k_minus"] = 1000      
CFG["confidence_p"] = 0.05 

In [ ]:
# ── 15. Run audit across all shadow models (fixed k) ─────────────────────────
M   = CFG["num_canaries"]
kp  = CFG["k_plus"]
km  = CFG["k_minus"]
r   = kp + km
p   = CFG["confidence_p"]
d   = CFG["delta"]

print(f"\nAudit params: m={M}, k+={kp}, k-={km}, r={r}, p={p}, δ={d}\n")

per_shadow_eps = []
for s in range(CFG["num_shadow"]):
    # Only score if we actually generated data for this shadow model
    if shadow_scores[s].abs().sum() == 0: continue 

    scores_s     = shadow_scores[s]
    member_s     = membership[s]
    sorted_idx   = torch.argsort(scores_s)

    bottom_k_idx = sorted_idx[:km]
    top_k_idx    = sorted_idx[-kp:]

    correct_out  = (~member_s[bottom_k_idx]).sum().item()
    correct_in   = member_s[top_k_idx].sum().item()
    v            = correct_in + correct_out

    acc = v / r
    eps_lb = get_eps_lower_bound(M, r, v, d, p) if v <= r <= M else 0.0
    per_shadow_eps.append(eps_lb)

    print(f"  shadow {s:2d} | correct_in={correct_in:3d}/{kp}  "
          f"correct_out={correct_out:3d}/{km}  acc={acc:.3f}  ε≥{eps_lb:.4f}")

if per_shadow_eps:
    print(f"\nBest ε lower bound (fixed k) : {max(per_shadow_eps):.4f}")



In [ ]:
# ── 16. Sweep k values to find optimal audit configuration ───────────────────
# 🟢 FIX: Allow sweep to go all the way up to M//2 (e.g., 2500)
print(f"\nSweeping k_plus = k_minus over [50, 100, …, {M//2}] …\n")

k_vals = list(range(50, (M // 2) + 1, 50))
sweep_table = []

for k in k_vals:
    k_best = 0.0
    for s in range(CFG["num_shadow"]):
        if shadow_scores[s].abs().sum() == 0: continue
        scores_s   = shadow_scores[s]
        member_s   = membership[s]
        sorted_idx = torch.argsort(scores_s)
        
        bot = sorted_idx[:k]
        top = sorted_idx[-k:]
        v   = (member_s[top].sum() + (~member_s[bot]).sum()).item()
        r_  = 2 * k
        
        if v <= r_ <= M:
            k_best = max(k_best, get_eps_lower_bound(M, r_, v, d, p))
    sweep_table.append((k, k_best))

sweep_k   = [t[0] for t in sweep_table]
sweep_eps = [t[1] for t in sweep_table]
best_idx  = int(np.argmax(sweep_eps))
best_k    = sweep_k[best_idx]
best_eps_sweep = sweep_eps[best_idx]

print(f"🏆 Optimal k={best_k}  →  ε ≥ {best_eps_sweep:.4f}")

---
## § 10 — Plots

In [ ]:
from pathlib import Path

# Add the output directory to your configuration
CFG["output_dir"] = Path("./audit_results")

# Create the folder if it doesn't exist yet
CFG["output_dir"].mkdir(parents=True, exist_ok=True)

print(f"Output directory set to: {CFG['output_dir']}")

In [ ]:
# ── 17. Figure 1: score distributions IN vs OUT ───────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

# 🟢 FIX: Calculate the missing summary variables from your existing arrays
best_eps = np.max(per_shadow_eps)
avg_eps = np.mean(per_shadow_eps)
best_k = sweep_k[np.argmax(sweep_eps)]

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

# ── (a) Score distribution IN vs OUT (averaged across shadow models) ──────────
all_in_scores  = shadow_scores[membership].numpy()
all_out_scores = shadow_scores[~membership].numpy()

ax = axes[0]
ax.hist(all_in_scores,  bins=50, alpha=0.65, color="steelblue",
        label=f"IN canaries  (n={len(all_in_scores):,})", density=True)
ax.hist(all_out_scores, bins=50, alpha=0.65, color="tomato",
        label=f"OUT canaries (n={len(all_out_scores):,})", density=True)
ax.axvline(0, color="black", lw=1, ls="--")
ax.set_title("Score Distribution: IN vs OUT", fontsize=12)
ax.set_xlabel("Score  =  loss(w₀, x) − loss(w_T, x)")
ax.set_ylabel("Density")
ax.legend(fontsize=9)
ax.text(0.97, 0.97,
        f"ΔMean = {all_in_scores.mean()-all_out_scores.mean():.3f}",
        ha="right", va="top", transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle="round", fc="white", alpha=0.7))

# ── (b) ε lower bound per shadow model ────────────────────────────────────────
ax = axes[1]
bar_colors = ["steelblue" if e == best_eps else "#aec6e8" for e in per_shadow_eps]
ax.bar(range(CFG["num_shadow"]), per_shadow_eps, color=bar_colors)
ax.axhline(avg_eps, color="darkorange", lw=1.5, ls="--", label=f"Mean ε={avg_eps:.3f}")
ax.axhline(best_eps, color="red", lw=1.5, ls=":", label=f"Best ε={best_eps:.3f}")
ax.set_title("ε Lower Bound per Shadow Model", fontsize=12)
ax.set_xlabel("Shadow Model Index")
ax.set_ylabel("Empirical ε lower bound")
ax.legend(fontsize=9)

# ── (c) Sweep: ε vs k ──────────────────────────────────────────────────────────
ax = axes[2]
ax.plot(sweep_k, sweep_eps, marker="o", ms=4, color="steelblue", lw=1.5)
ax.axvline(best_k, color="red", ls=":", lw=1.5, label=f"Best k={best_k}")
ax.set_title("ε Lower Bound vs Guess Budget k", fontsize=12)
ax.set_xlabel("k  (k_+ = k_- = k)")
ax.set_ylabel("ε lower bound")
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(CFG["output_dir"] / "audit_plots.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to", CFG["output_dir"] / "audit_plots.png")

---
## § 11 — Results Summary

In [ ]:
# ── 18. Final summary table ──────────────────────────────────────────────────
import json
import warnings
import numpy as np
import torch
from opacus import PrivacyEngine
from opacus.validators import ModuleValidator

# Compute theoretical ε using Opacus accountant with same hyperparameters.
_ds_tmp = NormDataset(data_gen.build_train_data(0), augment=False)
_ld_tmp = torch.utils.data.DataLoader(
    _ds_tmp, batch_size=CFG["batch_size"], shuffle=True, num_workers=0
)
_m_tmp   = build_model()
_m_tmp   = ModuleValidator.fix(_m_tmp)
_opt_tmp = torch.optim.SGD(_m_tmp.parameters(), lr=0.01)
_pe_tmp  = PrivacyEngine()
with warnings.catch_warnings():
    warnings.filterwarnings("ignore")
    _, _, _ld_acc = _pe_tmp.make_private(
        module=_m_tmp, optimizer=_opt_tmp, data_loader=_ld_tmp,
        noise_multiplier=CFG["noise_multiplier"],
        max_grad_norm=CFG["max_grad_norm"],
    )

# Advance accountant to match the number of gradient steps taken
_steps_per_epoch = len(_ld_acc)
for _ in range(CFG["num_epochs"] * _steps_per_epoch):
    _pe_tmp.accountant.step(
        noise_multiplier=CFG["noise_multiplier"],
        sample_rate=CFG["batch_size"] / len(_ds_tmp),
    )
theoretical_eps = _pe_tmp.get_epsilon(CFG["delta"])
del _m_tmp, _opt_tmp, _pe_tmp, _ld_tmp, _ld_acc, _ds_tmp

# 🟢 FIX: Calculate the missing accuracy and sweep summary variables
best_eps_sweep = np.max(sweep_eps)

per_shadow_acc = []
for i in range(CFG["num_shadow"]):
    # Use the median score of this model as the threshold for guessing IN vs OUT
    thresh = torch.median(shadow_scores[i]).item()
    in_correct = (shadow_scores[i][membership[i]] > thresh).sum().item()
    out_correct = (shadow_scores[i][~membership[i]] <= thresh).sum().item()
    acc = (in_correct + out_correct) / len(shadow_scores[i])
    per_shadow_acc.append(acc)

# ─────────────────────────────────────────────────────────────────────────────
DIV = "═" * 62
print(DIV)
print("   PRIVACY AUDIT RESULTS SUMMARY")
print(DIV)
print(f"\n  Paper: Steinke, Nasr & Jagielski (2023)")
print(f"\n── Configuration ──────────────────────────────────────────────")
print(f"  Canaries (m)            : {CFG['num_canaries']}")
print(f"  Canary type             : {CFG['canary_type']}")
print(f"  Shadow models           : {CFG['num_shadow']}")
print(f"  Epochs                  : {CFG['num_epochs']}")
print(f"  Batch size              : {CFG['batch_size']}")
print(f"  Noise multiplier σ      : {CFG['noise_multiplier']}")
print(f"  Max grad norm           : {CFG['max_grad_norm']}")
print(f"  δ                       : {CFG['delta']}")
print(f"  Guess budget k+ / k-    : {CFG['k_plus']} / {CFG['k_minus']}")
print(f"  Sweep best k            : {best_k}")
print(f"\n── Score Separation ────────────────────────────────────────────")
print(f"  IN  mean / std          : {all_in_scores.mean():+.4f} / {all_in_scores.std():.4f}")
print(f"  OUT mean / std          : {all_out_scores.mean():+.4f} / {all_out_scores.std():.4f}")
print(f"  Δ mean  (IN − OUT)      : {all_in_scores.mean()-all_out_scores.mean():+.4f}")
print(f"\n── Audit ───────────────────────────────────────────────────────")
print(f"  Audit accuracy (best)   : {max(per_shadow_acc):.4f}  (chance = 0.5000)")
print(f"  Audit accuracy (avg)    : {np.mean(per_shadow_acc):.4f}")
print(f"\n── Privacy Bounds ──────────────────────────────────────────────")
print(f"  ε upper bound (theory)  : ε ≤ {theoretical_eps:.4f}")
print(f"  ε lower bound (best)    : ε ≥ {best_eps:.4f}  (95 % confidence)")
print(f"  ε lower bound (sweep)   : ε ≥ {best_eps_sweep:.4f}  (best k={best_k})")
print(f"  ε lower bound (avg)     : ε ≥ {avg_eps:.4f}")
gap = theoretical_eps - best_eps
print(f"  Privacy gap             : [{best_eps:.4f}, {theoretical_eps:.4f}]  (Δ={gap:.4f})")
print(f"\n{DIV}")

result = dict(
    theoretical_eps=theoretical_eps,
    best_eps=float(best_eps),
    avg_eps=float(avg_eps),
    best_eps_sweep=float(best_eps_sweep),
    best_k=int(best_k),
    per_shadow_eps=[float(e) for e in per_shadow_eps],
    per_shadow_acc=[float(a) for a in per_shadow_acc],
    cfg={k: str(v) for k, v in CFG.items()},
)
out_path = CFG["output_dir"] / "audit_results.json"
with open(out_path, "w") as f:
    json.dump(result, f, indent=2)
print(f"Results saved → {out_path}")

---
## § 12 — Tips for Scaling Up

| Goal | Change |
|---|---|
| Reproduce paper ε≈100 | `num_shadow=32, num_epochs=60, noise_multiplier=0.4, batch_size=1024` |
| Reproduce paper ε≈7 | `num_shadow=32, num_epochs=220, noise_multiplier=3.0, batch_size=4096` |
| Faster training | Increase `batch_size`, `augmult_factor=16`, `num_workers=8` |
| Parallelise shadow models | Run cell below to train in parallel using `multiprocessing` or `concurrent.futures` |
| Better audit signal | Use `canary_type="blank_images"` (strongest signal for DP-SGD) |

In [ ]:
# ── 19. Optional: parallel training via ThreadPoolExecutor ───────────────────
# Uncomment to train all shadow models in parallel.
# Useful when num_shadow > num_GPUs and you have many CPU cores.
# Each model runs on DEVICE; for multi-GPU use torchrun instead.

# from concurrent.futures import ThreadPoolExecutor, as_completed
#
# def train_wrapper(s):
#     return s, train_shadow_model(s, CFG, data_gen, verbose=False)
#
# with ThreadPoolExecutor(max_workers=min(4, CFG["num_shadow"])) as pool:
#     futures = {pool.submit(train_wrapper, s): s for s in range(CFG["num_shadow"])}
#     for fut in tqdm.tqdm(as_completed(futures), total=CFG["num_shadow"]):
#         s, m = fut.result()
#         trained_models[s] = m
#         print(f"  shadow {s} done")
print("(Parallel training cell — uncomment to use)")